# 1kg_eur — rare-variant PCA with relateds removed

Diagnostic for the strange loadings in the rare-variant PCA. Relatedness is the
standard explanation: a family cluster is a set of individuals who share long
haplotypes, so they share the *same rare alleles* at the *same loci*. That is a
large, genuinely-correlated block in the genotype matrix, and PCA will happily
spend a component on it — producing a PC that a handful of people load extremely
on, driven by variants in a few genomic regions. Rare variants are much more
exposed to this than common ones, because a rare allele shared by 8 relatives is
most of that allele's variance in the whole sample.

The paper we are following (`s41467-026-73776-9`) excludes anyone related to
**third degree or closer** before fitting rare PCs. Our round-2 keep list does
not, because the rest of this pipeline *wants* relatives — they are the signal.

**Design: hold the variant set fixed, change only who is in the eigendecomposition.**
QC and pruning run once, on the full round-2 sample, exactly as the original run
did. Then PCA is fit twice — all samples, and unrelated-only — over the *same*
variants. If the loadings clean up in the second fit, relatedness was the cause.
If QC were redone within the unrelated subset, the variant set would change too
(allele frequencies shift, some variants leave the MAF band) and the comparison
would confound the two explanations. `REQC_IN_UNRELATED = True` adds that third
fit as a follow-up, once the main question is answered.

**Unrelated set:** built from the AoU aux relatedness table (`kin > 0.0442`,
i.e. third degree or closer), greedily pruned by repeatedly dropping the
individual with the most remaining relationships. No compute — this is the same
file `01_round1_gate.ipynb` reads. Alternatives considered and rejected:
`06a_close_pair_screen` covers only a_ij > 0.2 and lists both members of each
pair; nb07's KING output is 1st-degree only; the 0.1% GRM shards are on the 50K
subsample, not this sample; `plink2 --king-cutoff` is the O(N²) problem nb07 hit.

**Runs:** keep-list construction local (seconds). One Batch job for QC, pruning
and all PCA fits. Plots local.

## Config

In [ ]:
import os, subprocess
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SAMPLE_SET      = "1kg_eur"
CDR_VERSION     = "v9"
PROJECT_ID      = "wb-swift-sprout-7231"
REGION          = "us-central1"
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-swift-sprout-7231.iam.gserviceaccount.com"
NETWORK         = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK      = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"
CLOUD_SDK_TAG   = "581.0.0-slim"

WS_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
R_GS  = f"{WS_GS}/{SAMPLE_SET}"

# ── AoU aux relatedness (same file 01_round1_gate.ipynb reads) ────────────────
AUX      = os.path.expanduser(
    "~/workspace/cdrv9/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/aux")
REL_PATH = f"{AUX}/relatedness/samples_relatedness.tsv"
REL_COLS = ("i.s", "j.s", "kin")

# ── inputs ────────────────────────────────────────────────────────────────────
PANEL_GS  = f"{WS_GS}/01_ancestry_filtering/unified_panel/unified_panel_{CDR_VERSION}"
KEEP_GS   = f"{R_GS}/01_ancestry/round2/1kg_eur_keep_ids.txt"   # pre-CEUGBR-rename filename
PLINK2_GS = f"{R_GS}/03_grm/bin/plink2"

# ── outputs ───────────────────────────────────────────────────────────────────
OUT_GS      = f"{R_GS}/01_ancestry/rare_pca_unrelated"
LOG_GS      = f"{R_GS}/dsub_logs"
UNREL_GS    = f"{OUT_GS}/unrelated_keep_ids.txt"
LOCAL       = os.path.expanduser(f"~/scratch_{SAMPLE_SET}_rare_unrel")
os.makedirs(LOCAL, exist_ok=True)

# ── relatedness threshold ─────────────────────────────────────────────────────
# KING kinship. 1st degree ~0.25, 2nd ~0.125, 3rd ~0.0625. 0.0442 is the
# conventional midpoint between 3rd and 4th degree — "3rd degree or closer",
# matching the paper and plink2's --king-cutoff default reasoning.
KIN_CUT = 0.0442

# ── rare-variant band ─────────────────────────────────────────────────────────
# Panel floor is a pooled MAF 0.001, so the band is 0.1% <= MAF < 1% regardless.
RARE_MAF_MAX = 0.01
RARE_MAC_MIN = 51
GENO_MAX     = 0.10
RARE_PRUNE   = "10000kb 0.1"
N_PCS_FIT    = 20
REQC_IN_UNRELATED = True   # third fit: redo QC+pruning inside the unrelated subset

JOB_MACHINE, JOB_DISK_GB = "n1-highmem-32", 600

print(f"panel:     {PANEL_GS}")
print(f"keep:      {KEEP_GS}")
print(f"unrelated: {UNREL_GS}")
print(f"kin cut:   {KIN_CUT} (3rd degree or closer)")

## What already exists

If a rare-variant panel from an earlier run is already in the bucket, the job can
skip the 116 GB unified-panel download. This lists the candidates; set
`RARE_PFILE_GS` in the next cell if one of them is the panel your current PCA
used, otherwise leave it `None` and the job rebuilds from the unified panel.

In [ ]:
for d in (f"{R_GS}/01_ancestry/rare_variant_pca",
          f"{R_GS}/01_ancestry/ukb_style_pca",
          f"{R_GS}/03_grm/grm_input_0p1pct_50k",
          OUT_GS):
    print(f"\n── {d}")
    subprocess.run(["bash", "-c", f'gcloud storage ls "{d}/**" 2>/dev/null | head -20'],
                   check=False)

print(f"\n── aux relatedness")
print(f"{REL_PATH}: {'present' if os.path.isfile(REL_PATH) else 'MISSING'}")
if not os.path.isfile(REL_PATH):
    print(f"set REL_PATH/REL_COLS from what {os.path.dirname(REL_PATH)} holds:")
    subprocess.run(["bash", "-c", f'ls -la "{os.path.dirname(REL_PATH)}" 2>/dev/null'],
                   check=False)

In [ ]:
# Set to a pfile prefix in GCS to reuse an existing rare panel; None rebuilds
# from the unified panel. A reused panel must have been built on the SAME round-2
# sample, or the "all samples" fit is not the run you are trying to explain.
RARE_PFILE_GS = None

print("rebuilding from unified panel" if RARE_PFILE_GS is None
      else f"reusing {RARE_PFILE_GS}")

## Build the unrelated keep list

Greedy max-independent-set: while any relationship remains, drop the individual
involved in the most of them. Not optimal (the exact problem is NP-hard) but it
is what plink's `--king-cutoff` does, it is deterministic here, and it keeps far
more people than dropping both members of every pair.

Ties are broken on person_id so the list is reproducible across reruns.

In [ ]:
# round-2 keep list
KEEP_LOCAL = f"{LOCAL}/round2_keep.txt"
if not os.path.isfile(KEEP_LOCAL):
    subprocess.run(["gcloud", "storage", "cp", KEEP_GS, KEEP_LOCAL], check=True)

_lines = [l.strip() for l in open(KEEP_LOCAL) if l.strip()]
HDR    = _lines[0] if _lines[0].startswith("#") else None
cohort = set(_lines[1:] if HDR else _lines)
print(f"round-2 cohort: {len(cohort):,}")

assert os.path.isfile(REL_PATH), f"{REL_PATH} missing - see the discovery cell above"
i_col, j_col, k_col = REL_COLS
rel = pd.read_csv(REL_PATH, sep="\t", usecols=list(REL_COLS),
                  dtype={i_col: str, j_col: str})
print(f"aux relatedness rows: {len(rel):,}")

# pairs where BOTH members are in this cohort and kin exceeds the cut
rel = rel[(rel[k_col] > KIN_CUT)
          & rel[i_col].isin(cohort)
          & rel[j_col].isin(cohort)]
print(f"within-cohort pairs at kin > {KIN_CUT}: {len(rel):,}")
print(f"individuals involved in at least one: "
      f"{len(set(rel[i_col]) | set(rel[j_col])):,}")

In [ ]:
# greedy: repeatedly drop the highest-degree individual
adj = defaultdict(set)
for a, b in zip(rel[i_col], rel[j_col]):
    if a != b:
        adj[a].add(b)
        adj[b].add(a)

dropped = set()
while True:
    active = [(len(v), k) for k, v in adj.items() if v]
    if not active:
        break
    # max degree, tie-broken on id for reproducibility
    deg, worst = max(active, key=lambda t: (t[0], t[1]))
    dropped.add(worst)
    for nb in adj[worst]:
        adj[nb].discard(worst)
    adj[worst] = set()

unrelated = sorted(cohort - dropped)
print(f"dropped:   {len(dropped):,}")
print(f"unrelated: {len(unrelated):,} of {len(cohort):,} "
      f"({100 * len(unrelated) / len(cohort):.1f}%)")

# sanity: no surviving pair may remain
_surv = rel[~rel[i_col].isin(dropped) & ~rel[j_col].isin(dropped)]
assert len(_surv) == 0, f"{len(_surv)} related pairs survived the prune"
print("check: no related pair survives")

In [ ]:
UNREL_LOCAL = f"{LOCAL}/unrelated_keep_ids.txt"
with open(UNREL_LOCAL, "w") as fh:
    if HDR:
        fh.write(HDR + "\n")
    fh.write("\n".join(unrelated) + "\n")

subprocess.run(["gcloud", "storage", "cp", UNREL_LOCAL, UNREL_GS], check=True)
print(f"uploaded: {UNREL_GS}")
print(f"format matches the round-2 keep list (header: {HDR!r})")

## How related is the cohort, and to what degree

Worth a look before spending a job on it. If almost nobody is related, relatedness
is not what is producing the strange loadings and the rest of this notebook will
come back null — in which case suspect the usual alternatives: an unexcluded
long-range-LD region or the MHC (the rare arm excludes neither, deliberately),
or a batch/assay effect.

In [ ]:
bands = [(0.354, np.inf, "dup/MZ"), (0.177, 0.354, "1st"),
         (0.0884, 0.177, "2nd"), (0.0442, 0.0884, "3rd")]
rel_all = pd.read_csv(REL_PATH, sep="\t", usecols=list(REL_COLS),
                      dtype={i_col: str, j_col: str})
rel_all = rel_all[rel_all[i_col].isin(cohort) & rel_all[j_col].isin(cohort)]

rows = []
for lo, hi, name in bands:
    m = (rel_all[k_col] > lo) & (rel_all[k_col] <= hi)
    rows.append({"degree": name, "pairs": int(m.sum()),
                 "individuals": len(set(rel_all.loc[m, i_col])
                                    | set(rel_all.loc[m, j_col]))})
print(pd.DataFrame(rows).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(rel_all[k_col].clip(0, 0.6), bins=120, log=True)
for lo, _, name in bands:
    ax.axvline(lo, color="0.6", ls="--", lw=1)
    ax.text(lo, ax.get_ylim()[1], name, rotation=90, va="top", fontsize=8)
ax.set_xlabel("KING kinship"); ax.set_ylabel("pairs (log)")
ax.set_title(f"{SAMPLE_SET} round-2 — within-cohort relatedness")
plt.tight_layout(); plt.savefig(f"{LOCAL}/kinship_hist.png", dpi=150); plt.show()

## The Batch job

One task. QC and prune **once on the full round-2 sample**, then fit PCA on that
fixed variant set twice: `all` and `unrel`. With `REQC_IN_UNRELATED`, a third fit
`reqc` redoes QC and pruning inside the unrelated subset, which answers the
separate question of whether the variant set itself was being shaped by relatives.

`--freq counts` plus `allele-wts` on every fit, so loadings are comparable
between them.

In [ ]:
REQC_FLAG = "1" if REQC_IN_UNRELATED else "0"
PANEL_SRC = RARE_PFILE_GS if RARE_PFILE_GS else PANEL_GS

subprocess.run(["bash", "-c", f"""
# ALREADY RUN — remove the next line to resubmit
exit 0
set -eo pipefail
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E \
  "s|cloud-sdk:[0-9]+\\.[0-9]+\\.[0-9]+-slim|cloud-sdk:{CLOUD_SDK_TAG}|g" \
  "$DSUB_DIR/providers/google_utils.py"

dsub \
  --provider google-batch --project "{PROJECT_ID}" --regions "{REGION}" \
  --logging "{LOG_GS}" \
  --service-account "{SERVICE_ACCOUNT}" \
  --network "{NETWORK}" --subnetwork "{SUBNETWORK}" --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:{CLOUD_SDK_TAG}" \
  --name "rare-pca-unrel" \
  --machine-type "{JOB_MACHINE}" --disk-size "{JOB_DISK_GB}" \
  --input  PLINK2="{PLINK2_GS}" \
  --input  KEEP_ALL="{KEEP_GS}" \
  --input  KEEP_UNREL="{UNREL_GS}" \
  --env    PANEL_GS="{PANEL_SRC}" \
  --env    RARE_MAF_MAX="{RARE_MAF_MAX}" \
  --env    RARE_MAC_MIN="{RARE_MAC_MIN}" \
  --env    GENO_MAX="{GENO_MAX}" \
  --env    RARE_PRUNE="{RARE_PRUNE}" \
  --env    N_PCS="{N_PCS_FIT}" \
  --env    DO_REQC="{REQC_FLAG}" \
  --output-recursive OUT="{OUT_GS}" \
  --command '
    set -eo pipefail
    chmod +x "$PLINK2"
    NT=$(nproc)
    mkdir -p /mnt/data/panel /mnt/data/work "$OUT"
    cd /mnt/data/work

    echo "=== downloading panel"
    gcloud storage cp "$PANEL_GS".pgen "$PANEL_GS".pvar "$PANEL_GS".psam /mnt/data/panel/
    PANEL=/mnt/data/panel/$(basename "$PANEL_GS")

    echo "=== QC on the FULL round-2 sample (variant set held fixed)"
    "$PLINK2" --pfile "$PANEL" --keep "$KEEP_ALL" --nonfounders \
      --max-maf "$RARE_MAF_MAX" --mac "$RARE_MAC_MIN" \
      --geno "$GENO_MAX" --max-alleles 2 \
      --threads $NT --make-pgen --out rare_qc

    "$PLINK2" --pfile rare_qc --nonfounders \
      --indep-pairwise $RARE_PRUNE --threads $NT --out rare_prune

    echo "rare: $(grep -vc "^##" rare_qc.pvar) after QC, $(wc -l < rare_prune.prune.in) after pruning" \
      | tee -a "$OUT"/variant_counts.txt

    "$PLINK2" --pfile rare_qc --nonfounders --extract rare_prune.prune.in \
      --threads $NT --make-pgen --out rare_final

    # two fits over the SAME variants: all samples, then unrelated only
    for FIT in all unrel; do
      if [ "$FIT" = all ]; then KEEPFILE="$KEEP_ALL"; else KEEPFILE="$KEEP_UNREL"; fi
      echo "=== PCA fit: $FIT"
      "$PLINK2" --pfile rare_final --keep "$KEEPFILE" --nonfounders \
        --freq counts --pca approx "$N_PCS" allele-wts \
        --threads $NT --out fit_"$FIT"
      cp fit_"$FIT".eigenvec fit_"$FIT".eigenval "$OUT"/
      cp fit_"$FIT".eigenvec.allele "$OUT"/ || true
    done

    # third fit: QC and pruning redone inside the unrelated subset
    if [ "$DO_REQC" = "1" ]; then
      echo "=== re-QC inside the unrelated subset"
      "$PLINK2" --pfile "$PANEL" --keep "$KEEP_UNREL" --nonfounders \
        --max-maf "$RARE_MAF_MAX" --mac "$RARE_MAC_MIN" \
        --geno "$GENO_MAX" --max-alleles 2 \
        --threads $NT --make-pgen --out reqc_qc

      "$PLINK2" --pfile reqc_qc --nonfounders \
        --indep-pairwise $RARE_PRUNE --threads $NT --out reqc_prune

      echo "reqc: $(grep -vc "^##" reqc_qc.pvar) after QC, $(wc -l < reqc_prune.prune.in) after pruning" \
        | tee -a "$OUT"/variant_counts.txt

      "$PLINK2" --pfile reqc_qc --nonfounders --extract reqc_prune.prune.in \
        --freq counts --pca approx "$N_PCS" allele-wts \
        --threads $NT --out fit_reqc
      cp fit_reqc.eigenvec fit_reqc.eigenval "$OUT"/
      cp fit_reqc.eigenvec.allele "$OUT"/ || true
    fi

    echo "=== done"
    cat "$OUT"/variant_counts.txt
  '
"""], check=True)

## Monitor

In [ ]:
subprocess.run(["bash", "-c", f"""
dstat --provider google-batch \
  --project "{PROJECT_ID}" --location "{REGION}" \
  --jobs "rare-pca-unrel*" --status '*' --full 2>&1 | tail -40
"""], check=False)

## Download and load

In [ ]:
subprocess.run(["bash", "-c", f"""
set -eo pipefail
gcloud storage cp "{OUT_GS}/fit_*.eigenvec" "{OUT_GS}/fit_*.eigenval" \
  "{OUT_GS}/fit_*.eigenvec.allele" "{OUT_GS}/variant_counts.txt" "{LOCAL}/" 2>/dev/null
ls "{LOCAL}" | grep -E 'fit_|variant_counts'
"""], check=True)

print()
print(open(f"{LOCAL}/variant_counts.txt").read())

In [ ]:
PC_COLS = [f"PC{k}" for k in range(1, N_PCS_FIT + 1)]
FITS = ["all", "unrel"] + (["reqc"] if REQC_IN_UNRELATED else [])

def load_fit(fit):
    d = pd.read_csv(f"{LOCAL}/fit_{fit}.eigenvec", sep=r"\s+")
    idc = "#IID" if "#IID" in d.columns else "IID"
    d = d.rename(columns={idc: "person_id"})
    d["person_id"] = d["person_id"].astype(str)
    ev = np.loadtxt(f"{LOCAL}/fit_{fit}.eigenval")
    return d[["person_id"] + PC_COLS], ev / ev.sum() * 100

pcs, pct = {}, {}
for f in FITS:
    pcs[f], pct[f] = load_fit(f)
    print(f"{f:6s} n={len(pcs[f]):>7,}  " + "  ".join(f"PC{k+1} {pct[f][k]:.2f}%" for k in range(5)))

DROPPED = set(dropped)
print(f"\nrelateds dropped from the 'unrel' fit: {len(DROPPED):,}")

## Loadings — the actual question

A loading spike means one PC is being carried by variants in a narrow genomic
region rather than by genome-wide structure. Plotted as |weight| against variant
index, so a spike is a visible vertical stripe.

If the `all` panel spikes and `unrel` does not, relatedness was the cause.

In [ ]:
def load_weights(fit):
    L = pd.read_csv(f"{LOCAL}/fit_{fit}.eigenvec.allele", sep=r"\s+")
    idc = "#ID" if "#ID" in L.columns else "ID"
    parts = L[idc].str.split(":", n=2, expand=True)
    L["CHROM"] = parts[0]
    L["POS"] = pd.to_numeric(parts[1], errors="coerce")
    return L.dropna(subset=["POS"])

W = {f: load_weights(f) for f in FITS if os.path.isfile(f"{LOCAL}/fit_{f}.eigenvec.allele")}

n_show = 5
fig, axes = plt.subplots(n_show, len(W), figsize=(6 * len(W), 2.2 * n_show),
                         squeeze=False, sharex="col")
for col, f in enumerate(W):
    for row in range(n_show):
        ax = axes[row, col]
        ax.scatter(range(len(W[f])), W[f][f"PC{row+1}"].abs(),
                   s=0.3, alpha=0.4, rasterized=True)
        ax.set_ylabel(f"PC{row+1}")
        if row == 0:
            ax.set_title(f"{f}  (n={len(pcs[f]):,})")
plt.suptitle("rare-variant PCA loadings — |weight| by variant index")
plt.tight_layout(); plt.savefig(f"{LOCAL}/loadings_compare.png", dpi=150); plt.show()

In [ ]:
# quantify: how concentrated is each PC's loading mass?
# Top-1% share = fraction of sum(w^2) carried by the top 1% of variants.
# Diffuse genome-wide structure sits near 1% (uniform); a spike pushes it up.
rows = []
for f in W:
    for k in range(1, N_PCS_FIT + 1):
        w2 = W[f][f"PC{k}"].to_numpy() ** 2
        n1 = max(1, len(w2) // 100)
        top = np.sort(w2)[-n1:].sum() / w2.sum()
        rows.append({"fit": f, "PC": k, "top1pct_share": top,
                     "pct_var": pct[f][k - 1]})
conc = pd.DataFrame(rows).pivot(index="PC", columns="fit", values="top1pct_share")
print("share of squared loading mass in the top 1% of variants (uniform = 0.01):")
print(conc.round(3).to_string())

## Are the odd PCs driven by the relatives specifically?

Direct test, independent of the loadings. In the `all` fit, score each PC by how
much of its variance sits in the dropped (related) individuals. Relatives are a
small fraction of the sample, so under genome-wide structure they should carry
their population share of the variance and no more. A PC where they carry far
more than that is a family-cluster PC.

In [ ]:
d = pcs["all"].copy()
d["is_rel"] = d["person_id"].isin(DROPPED)
frac_rel = d["is_rel"].mean()
print(f"related individuals in the 'all' fit: {d['is_rel'].sum():,} ({100*frac_rel:.2f}%)")

rows = []
for k in range(1, N_PCS_FIT + 1):
    v = d[f"PC{k}"].to_numpy()
    v = v - v.mean()
    ss_tot = (v ** 2).sum()
    ss_rel = (v[d["is_rel"].to_numpy()] ** 2).sum()
    # how many of the 20 most extreme individuals on this PC are relatives
    top20 = d.iloc[np.argsort(-np.abs(v))[:20]]
    rows.append({"PC": k, "pct_var": pct["all"][k - 1],
                 "var_share_in_relateds": ss_rel / ss_tot,
                 "expected_if_diffuse": frac_rel,
                 "relateds_in_top20": int(top20["is_rel"].sum())})
fam = pd.DataFrame(rows).set_index("PC")
print()
print(fam.round(4).to_string())
print(f"\nPCs where relateds carry >5x their share: "
      f"{list(fam.index[fam['var_share_in_relateds'] > 5 * frac_rel])}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
ax.bar(fam.index, fam["var_share_in_relateds"])
ax.axhline(frac_rel, color="crimson", ls="--", lw=1,
           label=f"expected if diffuse ({frac_rel:.3f})")
ax.set_xlabel("PC"); ax.set_ylabel("share of PC variance in relateds")
ax.set_xticks(range(1, N_PCS_FIT + 1)); ax.legend(fontsize=8)
ax.set_title("'all' fit — is the PC carried by relatives?")

ax = axes[1]
for f, c in zip(FITS, ("tab:blue", "tab:orange", "tab:green")):
    ax.plot(range(1, N_PCS_FIT + 1), pct[f][:N_PCS_FIT], "-o", ms=4, color=c, label=f)
ax.set_xlabel("PC"); ax.set_ylabel("% variance explained")
ax.set_xticks(range(1, N_PCS_FIT + 1)); ax.legend()
ax.set_title("scree by fit")
plt.tight_layout(); plt.savefig(f"{LOCAL}/relatedness_diagnostic.png", dpi=150); plt.show()

## PC pairs, relatives highlighted

The visual version: if a PC is a family artefact, a handful of coloured points
sit far out on their own while the cloud stays put.

In [ ]:
rng = np.random.default_rng(42)
sub = rng.choice(len(d), size=min(40_000, len(d)), replace=False)
ds = d.iloc[sub]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (i, j) in zip(axes.flat, [(1,2),(3,4),(5,6),(1,3),(2,4),(7,8)]):
    ax.scatter(ds.loc[~ds["is_rel"], f"PC{i}"], ds.loc[~ds["is_rel"], f"PC{j}"],
               s=1, alpha=0.1, color="0.75", rasterized=True, label="unrelated")
    r = d[d["is_rel"]]
    ax.scatter(r[f"PC{i}"], r[f"PC{j}"], s=3, alpha=0.5, color="crimson",
               rasterized=True, label="related (dropped)")
    ax.set_xlabel(f"PC{i}"); ax.set_ylabel(f"PC{j}")
axes.flat[0].legend(fontsize=8, markerscale=4)
plt.suptitle("'all' fit — relateds in red")
plt.tight_layout(); plt.savefig(f"{LOCAL}/pc_pairs_relateds.png", dpi=150); plt.show()

## Do the two fits recover the same axes?

Scoring everyone through both sets of allele weights would be the cleaner
comparison, but the fits share individuals (the unrelated set is a subset), so
correlating PCs over those shared individuals is enough. PCs are identified only
up to sign and up to rotation within near-degenerate subspaces, so report
`max |r|` against all PCs of the other fit, not just the diagonal.

In [ ]:
a = pcs["all"].set_index("person_id")[PC_COLS]
for f in FITS[1:]:
    b = pcs[f].set_index("person_id")[PC_COLS]
    ids = a.index.intersection(b.index)
    R = np.corrcoef(a.loc[ids].to_numpy().T, b.loc[ids].to_numpy().T)[:N_PCS_FIT, N_PCS_FIT:]
    t = pd.DataFrame({
        "PC": range(1, N_PCS_FIT + 1),
        "diag_abs_r": np.abs(np.diag(R)),
        "max_abs_r": np.abs(R).max(axis=1),
        "best_match": np.abs(R).argmax(axis=1) + 1,
    }).set_index("PC")
    print(f"\n── all vs {f} (over {len(ids):,} shared individuals)")
    print(t.round(3).to_string())

## Summary and upload

In [ ]:
summary = fam.join(conc.add_prefix("top1pct_"))
SUMMARY = f"{LOCAL}/rare_pca_unrelated_summary.tsv"
summary.to_csv(SUMMARY, sep="\t", float_format="%.5f")
print(summary.round(4).to_string())

subprocess.run(["bash", "-c", f"""
set -eo pipefail
gcloud storage cp "{SUMMARY}" "{OUT_GS}/rare_pca_unrelated_summary.tsv"
gcloud storage cp "{LOCAL}/"*.png "{OUT_GS}/figures/"
"""], check=True)
print(f"\nuploaded to {OUT_GS}")

## Copy notebook to bucket

In [ ]:
_nb = os.path.expanduser('~/repos/AOU-covariance/notebooks/extra/rare_pca_unrelated.ipynb')
_gs = f'{OUT_GS}/notebooks/rare_pca_unrelated.ipynb'
subprocess.run(['gcloud', 'storage', 'cp', _nb, _gs], check=True)
print(f'notebook -> {_gs}')